# ☁️ AI Dubbing Cloud GPU Server
**Run this on Colab → Your local IDE uses Colab's T4 GPU automatically!**

---
### How it works:
1. This notebook starts a GPU-powered API server on Colab
2. ngrok creates a public URL for it
3. You paste that URL in your local `.env` file
4. Your local `node backend/server.js` sends videos here for processing
5. XTTS clones the original voice using Colab's 16GB VRAM!
---

## Step 1: Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\n✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2: Install Dependencies (~3-5 min)

In [ ]:
# Install system dependencies
!sudo apt-get -y install espeak-ng libsndfile1-dev

# Install the community-maintained TTS fork (supports Python 3.12)
!pip install -q coqui-tts
!pip install -q librosa soundfile

# Install other dependencies (force update google-generativeai)
!pip install -q faster-whisper deep-translator ffmpeg-python edge-tts gtts demucs flask pyngrok
!pip install -q -U google-generativeai

print("\n✅ All packages installed!")

## Step 3: Set Your ngrok Token
Go to ngrok.com → Sign up (free)
Go to Dashboard → Your Authtoken
Paste it below

In [ ]:
NGROK_TOKEN = "39WInaWzPKmQ23KSVYQkyxET9ch_2pcKkDDo1P784NDxndzMV"  # <-- PASTE YOUR NGROK TOKEN HERE

if not NGROK_TOKEN:
    print("❌ Please paste your ngrok token above!")
    print("   Get it free from: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    print("✅ Token set!")

## Step 3.5: Install Heavy Dependencies on Colab

In [ ]:
# 1. Install Heavy Dependencies on Colab
!pip install -q rvc-python gfpgan insightface ffmpeg-python

# 2. Fix the pip metadata bug for RVC
!python -m pip install "pip<24.1"
!pip install -q rvc-python
!python -m pip install --upgrade pip

# 3. Clone and Install LatentSync
!git clone https://github.com/bytedance/LatentSync.git latentsync
!cd latentsync && pip install -q -r requirements.txt
!sed -i 's/mediapipe==0.10.11/mediapipe>=0.10.14/g' latentsync/requirements.txt


## Step 4: Upload cloud_api.py
Upload the `cloud_api.py` file from your project's `python/` folder.

In [ ]:
from google.colab import files
print("📁 Upload cloud_api.py from your project:")
uploaded = files.upload()
print(f"\n✅ Uploaded: {list(uploaded.keys())}")

## Step 5: Start the Cloud API Server 🚀
This will:
1. Pre-load XTTS model (~2 min first time)
2. Start Flask API on port 5050
3. Create ngrok tunnel
4. Print the **PUBLIC URL** you need to copy

In [ ]:
import subprocess
import threading
import time
from pyngrok import ngrok

# Ensure NGROK_TOKEN is available
try:
    print(f"Using token starting with: {NGROK_TOKEN[:5]}...")
except NameError:
    print("❌ Error: NGROK_TOKEN is not defined. Please run Step 3 (cell S0n2v6rIhhYr) first!")
    raise

# Kill existing ngrok processes to prevent 'too many tunnels' error
ngrok.kill()

# Set ngrok token
ngrok.set_auth_token(NGROK_TOKEN)

# Start ngrok tunnel
public_url = ngrok.connect(5050)
print("\n" + "="*60)
print(f"🌐 YOUR CLOUD API URL: {public_url}")
print("="*60)
print("\n📋 COPY THE URL ABOVE AND:")
print("   1. Create/edit file: d:\\newProject\\.env")
print(f"   2. Add this line: CLOUD_API_URL={public_url}")
print("   3. Restart your local server: node backend/server.js")
print("   4. Upload a video on localhost:5000 → It processes HERE on Colab GPU! 🚀")
print("\n" + "="*60)
print("⏳ Starting Flask server (loading XTTS model)...")
print("   This will take ~2-3 minutes on first run.")
print("   Keep this tab open! Closing it stops the server.")
print("="*60 + "\n")

# Run Flask server (using the correct filename found in /content/)
!python "cloud_api.py"